# 04 - Destroy Infrastructure

Use this notebook to clean up all resources created by the demo.

**Why this exists:** prototype demos should include a clear teardown path to avoid ongoing charges.

---

## Step 1 - Load environment and resolve Azure CLI

In [8]:
import json
import os
import re
import shutil
import subprocess
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(dotenv_path="../env/.env")

def resolve_az_cli():
    az_in_path = shutil.which("az")
    if az_in_path:
        return az_in_path
    windows_fallbacks = [
        Path(r"C:\\Program Files\\Microsoft SDKs\\Azure\\CLI2\\wbin\\az.cmd"),
        Path(r"C:\\Program Files (x86)\\Microsoft SDKs\\Azure\\CLI2\\wbin\\az.cmd"),
    ]
    for candidate in windows_fallbacks:
        if candidate.exists():
            return str(candidate)
    return None

def extract_resource_group_from_deploy_notebook():
    deploy_nb = Path("../notebooks/01_deploy_infra.ipynb")
    if not deploy_nb.exists():
        return None
    data = json.loads(deploy_nb.read_text(encoding="utf-8"))
    for cell in data.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        source = "\n".join(cell.get("source", []))
        match = re.search(r'^RESOURCE_GROUP\s*=\s*"([^"]+)"', source, re.MULTILINE)
        if match:
            return match.group(1)
    return None

az_cmd = resolve_az_cli()
if not az_cmd:
    raise RuntimeError("Azure CLI not found. Install Azure CLI and rerun.")

resource_group = os.getenv("AZURE_RESOURCE_GROUP") or extract_resource_group_from_deploy_notebook()
if not resource_group:
    raise RuntimeError(
        "Could not determine resource group. Set AZURE_RESOURCE_GROUP in ../env/.env "
        "or define RESOURCE_GROUP in 01_deploy_infra.ipynb."
    )

exists_check = subprocess.run(
    [az_cmd, "group", "exists", "--name", resource_group],
    capture_output=True,
    text=True,
    check=True,
 )
if exists_check.stdout.strip().lower() != "true":
    raise RuntimeError(f"Resource group '{resource_group}' does not exist in the current subscription.")

print(f"Using Azure CLI: {az_cmd}")
print(f"Resource group to delete: {resource_group}")

Using Azure CLI: C:\Program Files\Microsoft SDKs\Azure\CLI2\wbin\az.CMD
Resource group to delete: rg-speech01-foundry


## Step 2 - Review resources before deletion

In [9]:
subprocess.run([az_cmd, "resource", "list", "--resource-group", resource_group, "--output", "table"], check=False)

CompletedProcess(args=['C:\\Program Files\\Microsoft SDKs\\Azure\\CLI2\\wbin\\az.CMD', 'resource', 'list', '--resource-group', 'rg-speech01-foundry', '--output', 'table'], returncode=0)

## Step 3 - Confirm and delete

Set `CONFIRM_DELETE = True` only when you are ready.

In [11]:
CONFIRM_DELETE = True

if not CONFIRM_DELETE:
    print("Deletion skipped. Set CONFIRM_DELETE = True to proceed.")
else:
    subprocess.run([az_cmd, "group", "delete", "--name", resource_group, "--yes", "--no-wait"], check=True)
    print(f"Deletion triggered for resource group: {resource_group}")

Deletion triggered for resource group: rg-speech01-foundry


## Step 4 - Purge soft-deleted AI resources (optional)

Set the confirmation flag in the next cell only if you want to permanently purge soft-deleted Cognitive Services accounts.

This is useful when you need the account name to be reusable immediately.

In [15]:
from urllib.parse import urlparse

CONFIRM_PURGE_SOFT_DELETES = True

ai_endpoint = os.getenv("AZURE_AI_ENDPOINT", "").strip()
target_account_name = ""
if ai_endpoint:
    target_account_name = urlparse(ai_endpoint).netloc.split(".")[0]

deleted_result = subprocess.run(
    [az_cmd, "cognitiveservices", "account", "list-deleted", "--output", "json"],
    capture_output=True,
    text=True,
    check=True,
 )
deleted_accounts = json.loads(deleted_result.stdout or "[]")

if target_account_name:
    candidates = [
        item for item in deleted_accounts
        if str(item.get("name", "")).lower() == target_account_name.lower()
    ]
else:
    candidates = deleted_accounts

if not candidates:
    if target_account_name:
        print(f"No soft-deleted Cognitive Services account found for: {target_account_name}")
    else:
        print("No soft-deleted Cognitive Services accounts found.")
elif not CONFIRM_PURGE_SOFT_DELETES:
    print("Purge skipped. Set CONFIRM_PURGE_SOFT_DELETES = True to permanently purge:")
    for item in candidates:
        print(f"- {item.get('name')} ({item.get('location')})")
else:
    for item in candidates:
        name = item.get("name")
        location = item.get("location")
        if not name or not location:
            print(f"Skipping invalid entry: {item}")
            continue

        purge_result = subprocess.run(
            [
                az_cmd,
                "cognitiveservices",
                "account",
                "purge",
                "--name",
                name,
                "--location",
                location,
                "--resource-group",
                resource_group,
            ],
            capture_output=True,
            text=True,
            check=False,
        )
        if purge_result.returncode != 0:
            raise RuntimeError(
                f"Failed to purge soft-deleted Cognitive Services account '{name}'.\n"
                f"stderr: {purge_result.stderr.strip()}\n"
                f"stdout: {purge_result.stdout.strip()}"
            )
        print(f"Purged soft-deleted Cognitive Services account: {name} ({location})")

Purged soft-deleted Cognitive Services account: aispeech016m4wo (eastus2)
